# 09a — Memory and serving performance — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/09.md) · [Course map](../PLAN.md) · [Project](../../projects/stage09/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Estimate weight and KV memory
- Distinguish latency from throughput
- Use measurements to choose serving settings


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Memory budgets and bottlenecks

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A bit is a binary storage unit; eight bits make a byte. GiB means 2**30 bytes. Weight memory is approximately parameter_count * bits_per_parameter / 8. This excludes runtime structures, activations, and the attention KV cache.

KV storage grows with sequence length, layer count, KV heads, head dimension, batch/concurrent sequences, and stored precision. A grouped-query architecture can use fewer KV heads than query heads, so use the actual configuration rather than guessing from model parameter count.

Latency is elapsed time for a request. Throughput is completed work per wall-clock time. A bottleneck is the resource limiting progress, such as memory bandwidth, compute, queueing, or I/O. Theoretical storage estimates must be compared with measurements.

### Worked example: follow the values

One billion 16-bit weights need 2 billion bytes, about 1.86 GiB. Two tensors per layer store keys and values. Doubling context length roughly doubles that cache component; doubling concurrent sequences can do the same. A model whose weights fit can still exhaust memory during long-context requests.

### Pause and explain

Why is a smaller quantized file not a guarantee of lower latency?

<details><summary>Check your reasoning after trying</summary>

The runtime may use different kernels or incur conversion overhead; hardware and workload determine measured speed.

</details>

### Common mistakes to check

Distinguish bytes from bits and GB from GiB. Record the assumptions attached to every estimate.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

Weight memory is roughly parameter count × bits per parameter / 8. This estimate excludes quantization metadata, activations, temporary workspaces, runtime overhead, and KV cache. FP16 and BF16 both use two bytes but have different numerical ranges. CUDA is NVIDIA’s compute platform; other devices use other backends. A model fitting in RAM does not imply acceptable latency.

For a decoder, KV cache storage roughly scales with 2 × layers × KV heads × head dimension × sequence length × batch × bytes. Grouped-query attention can reduce KV heads relative to query heads. Longer contexts and more concurrent users can dominate memory.

Latency measures a request’s elapsed time; throughput measures completed work per unit time. Report time to first token, inter-token latency, tokens/second, concurrency, prompt/output lengths, and p50/p95. Batching amortizes work but can increase waiting time. Continuous batching allows sequences to enter/leave a running batch. Do not select hardware from weight size alone.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
print("7 billion FP16 weights, GiB:", 7_000_000_000*2/1024**3)


## Exercise 1: Weight estimate

Return GiB for parameters and bits; require positive values.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def weight_gib(parameters, bits):
    if parameters <= 0 or bits <= 0: raise ValueError("Positive sizes required")
    return parameters*bits/8/1024**3


In [ ]:
assert weight_gib(1024**3, 8) == 1
assert weight_gib(1024**3, 4) == .5
print("Exercise 1: checks passed")


**Why this works:** Always label decimal GB versus binary GiB to avoid unit errors.


## Exercise 2: KV estimate

Return bytes for layers, kv_heads, head_dim, tokens, batch=1, bytes_per_value=2.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def kv_bytes(layers, kv_heads, head_dim, tokens, batch=1, bytes_per_value=2):
    return 2*layers*kv_heads*head_dim*tokens*batch*bytes_per_value


In [ ]:
assert kv_bytes(2, 4, 8, 16) == 4096
print("Exercise 2: checks passed")


**Why this works:** This simplified estimate omits block allocation and implementation overhead.


## Exercise 3: Nearest-rank percentile

Return nearest-rank percentile p in (0,1] of nonempty values.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def percentile(values, p):
    import math
    if not values or not 0 < p <= 1: raise ValueError("Invalid percentile")
    return sorted(values)[math.ceil(p*len(values))-1]


In [ ]:
assert percentile(list(range(1, 101)), .95) == 95
assert percentile([3], .5) == 3
print("Exercise 3: checks passed")


**Why this works:** Percentile conventions differ; document yours and collect enough observations for tail estimates.


## Independent transfer

Create a hardware budget for 1B, 7B, and 14B models at 4/8/16 bits and multiple context lengths. Measure your actual machine before buying hardware or choosing a deployment target.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [09a interface and acceptance cases](../assignments/09a.md).
2. Copy the [blank starter](../assignments/starters/09a.py) to `work/assignments/09a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 09a --solution work/assignments/09a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/09a.md#09a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 09a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why does concurrency consume memory?

   Each active sequence needs state such as KV cache in addition to shared model weights.

2. Is quantization always faster?

   No. Speed depends on hardware support, kernels, batching, and memory/compute bottlenecks.


## Reading and review

- [Primary reference 1](https://docs.vllm.ai/en/latest/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
